# Milestone 1.5: Single-Compound ADMET Summary Report
Generate a structured safety and drug-likeness profile for a single compound.
Defines the API response schema used by the production backend.

In [ ]:
import os, re, glob, json, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import SVG, HTML, display
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from captum.attr import LayerIntegratedGradients
warnings.filterwarnings('ignore')

# ── Constants (same as M2, M3, M4) ─────────────────────────────────────────
BASE_MODEL = 'DeepChem/ChemBERTa-77M-MTR'
checkpoints = sorted(glob.glob('chemberta-tox21-multitarget-*'))
MODEL_DIR = checkpoints[-1] if checkpoints else 'chemberta-tox21-multitarget'
NUM_TARGETS = 12
TARGET_NAMES = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
]
TARGET_IDX = {name: i for i, name in enumerate(TARGET_NAMES)}
THRESHOLDS = {
    'NR-AR': 0.95, 'NR-AR-LBD': 0.95, 'NR-AhR': 0.75,
    'NR-Aromatase': 0.85, 'NR-ER': 0.70, 'NR-ER-LBD': 0.85,
    'NR-PPAR-gamma': 0.85, 'SR-ARE': 0.65, 'SR-ATAD5': 0.80,
    'SR-HSE': 0.85, 'SR-MMP': 0.85, 'SR-p53': 0.85,
}
SEVERITY_WEIGHTS = {
    'NR-AR': 1.0, 'NR-AR-LBD': 1.0, 'NR-AhR': 1.5,
    'NR-Aromatase': 1.0, 'NR-ER': 1.0, 'NR-ER-LBD': 1.0,
    'NR-PPAR-gamma': 1.0, 'SR-ARE': 1.0, 'SR-ATAD5': 1.0,
    'SR-HSE': 1.0, 'SR-MMP': 1.5, 'SR-p53': 1.5,
}
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
WEIGHT_ARRAY = np.array([SEVERITY_WEIGHTS[t] for t in TARGET_NAMES])
WEIGHT_SUM   = WEIGHT_ARRAY.sum()
print(f"Checkpoint: {MODEL_DIR}  Device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_TARGETS,
    ignore_mismatched_sizes=True, attn_implementation='eager',
)
model = PeftModel.from_pretrained(base, MODEL_DIR).merge_and_unload()
model.eval().to(DEVICE)

def _forward_for_ig(input_ids, attention_mask, target_idx):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    return torch.sigmoid(out.logits[:, target_idx])

EMBED_LAYER = (model.roberta.embeddings if hasattr(model, 'roberta')
               else model.bert.embeddings)
lig = LayerIntegratedGradients(_forward_for_ig, EMBED_LAYER)
print("Model and IG ready.")

In [ ]:
_ATOM_RE = re.compile(r'\[[^\]]+\]|Cl|Br|[BCNOPSFI]|[bcnops]')

_pains_params = FilterCatalogParams()
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
_pains_catalog = FilterCatalog(_pains_params)

def mol_to_svg(mol, width=360, height=260):
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.drawOptions().addStereoAnnotation = False
    drawer.DrawMolecule(mol)
    finish = getattr(drawer, 'FinishDrawing', None) or getattr(drawer, 'EndDrawing')
    finish()
    return drawer.GetDrawingText()

def compute_admet(mol):
    mw  = Descriptors.ExactMolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = Descriptors.NumHDonors(mol)
    hba  = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rb   = Descriptors.NumRotatableBonds(mol)
    violations = int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)
    pains = [e.GetDescription() for e in _pains_catalog.GetMatches(mol)]
    return {
        "molecular_weight": round(mw, 2),
        "logp": round(logp, 2),
        "hbd": hbd, "hba": hba,
        "tpsa": round(tpsa, 2),
        "rotatable_bonds": rb,
        "lipinski_pass": violations <= 1,
        "veber_pass": tpsa <= 140 and rb <= 10,
        "pains_alerts": pains,
    }

In [ ]:
# === ASSERTION CELL ===
# Aspirin: MW=180, logP=1.2, HBD=1, HBA=3 — should pass Lipinski and Veber
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
admet_asp = compute_admet(aspirin)
assert admet_asp['lipinski_pass'], "Aspirin should pass Lipinski"
assert admet_asp['veber_pass'],    "Aspirin should pass Veber"
assert admet_asp['molecular_weight'] < 200, f"MW wrong: {admet_asp['molecular_weight']}"
assert admet_asp['pains_alerts'] == [], f"Aspirin should have no PAINS alerts"

# Cyclosporin A: MW~1202, should fail Lipinski
cyclosporin = Chem.MolFromSmiles(
    'CCC1C(=O)N(CC(=O)N(C(C(=O)NC(C(=O)N(C(C(=O)NC(C(=O)NC(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N1C)CC(C)C)C)CC(C)C)C)C)CC(C)C)C)C(C)C)C)CC(C)C)C)C(C)C)C)C'
)
if cyclosporin:
    admet_cyc = compute_admet(cyclosporin)
    assert not admet_cyc['lipinski_pass'], "Cyclosporin should fail Lipinski (MW > 500)"

print("✓ ADMET assertions passed")
print(f"  Aspirin: MW={admet_asp['molecular_weight']}, logP={admet_asp['logp']}, "
      f"Lipinski={admet_asp['lipinski_pass']}, Veber={admet_asp['veber_pass']}")

In [ ]:
def predict_probs(smiles):
    """Single compound → list of 12 floats."""
    enc = tokenizer(smiles, return_tensors='pt', truncation=True, max_length=512)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits[0]
    return torch.sigmoid(logits).cpu().tolist()

def compute_explainability(canonical, mol, probs):
    """Run IG for top-3 targets by probability. Returns per-target atom scores."""
    enc = tokenizer(canonical, return_tensors='pt', return_offsets_mapping=True,
                    truncation=True, max_length=512)
    input_ids = enc['input_ids'].to(DEVICE)
    attn_mask = enc['attention_mask'].to(DEVICE)
    offsets   = enc['offset_mapping'][0].tolist()
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)

    positions = [(m.start(), m.end()) for m in _ATOM_RE.finditer(canonical)]
    a_map = {}
    for atom_idx, (a_start, _) in enumerate(positions):
        for tok_idx, (t_start, t_end) in enumerate(offsets):
            if t_start <= a_start < t_end:
                a_map[atom_idx] = tok_idx
                break

    top_target_indices = np.argsort(probs)[-3:][::-1]
    result = {}
    for idx in top_target_indices:
        target_name = TARGET_NAMES[idx]
        attrs, _ = lig.attribute(
            inputs=input_ids,
            baselines=baseline_ids,
            additional_forward_args=(attn_mask, int(idx)),
            n_steps=30,
            return_convergence_delta=True,
        )
        token_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()
        atom_scores = {
            i: float(token_scores[a_map[i]])
            if i in a_map and a_map[i] < len(token_scores) else 0.0
            for i in range(mol.GetNumAtoms())
        }
        sorted_atoms = sorted(atom_scores.items(), key=lambda x: x[1], reverse=True)
        top_atoms = [k for k, v in sorted_atoms[:5] if v > 0]
        result[target_name] = {
            "atom_scores": {str(k): round(v, 4) for k, v in atom_scores.items()},
            "top_atoms": top_atoms,
        }
    return result

In [ ]:
def generate_report(smiles: str, compound_name: str = "") -> dict:
    """
    Source of truth for the API response schema.
    Returns a structured compound safety profile.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Cannot parse SMILES: {smiles}")
    canonical = Chem.MolToSmiles(mol)
    mol_can   = Chem.MolFromSmiles(canonical)

    probs = predict_probs(canonical)

    toxicity = {
        name: {
            "probability": round(probs[i], 4),
            "label": "TOXIC" if probs[i] >= THRESHOLDS[name] else "safe",
            "threshold": THRESHOLDS[name],
        }
        for i, name in enumerate(TARGET_NAMES)
    }

    admet = compute_admet(mol_can)
    explainability = compute_explainability(canonical, mol_can, probs)

    risk_score = float(np.dot(probs, WEIGHT_ARRAY) / WEIGHT_SUM)
    tier = "High" if risk_score > 0.5 else "Moderate" if risk_score > 0.25 else "Low"
    toxic_targets = [name for name, t in toxicity.items() if t["label"] == "TOXIC"]

    return {
        "smiles": smiles,
        "canonical_smiles": canonical,
        "compound_name": compound_name,
        "structure_svg": mol_to_svg(mol_can),
        "toxicity": toxicity,
        "admet": admet,
        "explainability": explainability,
        "risk_summary": {
            "composite_score": round(risk_score, 4),
            "tier": tier,
            "toxic_targets": toxic_targets,
            "flagged_targets": admet["pains_alerts"],
        },
    }

In [ ]:
# === ASSERTION CELL ===
test_report = generate_report('CC(=O)Oc1ccccc1C(=O)O', 'Aspirin')

REQUIRED_KEYS = {'smiles', 'canonical_smiles', 'compound_name', 'structure_svg',
                 'toxicity', 'admet', 'explainability', 'risk_summary'}
assert REQUIRED_KEYS == set(test_report.keys()), f"Missing keys: {REQUIRED_KEYS - set(test_report.keys())}"
assert len(test_report['toxicity']) == 12, "Expected 12 toxicity targets"
assert test_report['admet']['lipinski_pass'] == True, "Aspirin should pass Lipinski"
assert test_report['risk_summary']['tier'] in ('Low', 'Moderate', 'High'), "Invalid tier"
assert test_report['structure_svg'].startswith('<svg'), "SVG should start with <svg"
assert isinstance(test_report['risk_summary']['composite_score'], float)
assert 0 <= test_report['risk_summary']['composite_score'] <= 1

print(f"✓ generate_report() schema assertions passed")
print(f"  Aspirin: tier={test_report['risk_summary']['tier']}, "
      f"score={test_report['risk_summary']['composite_score']:.3f}, "
      f"toxic_targets={test_report['risk_summary']['toxic_targets']}")

In [ ]:
def render_report_html(report):
    """Render a generate_report() dict as rich HTML."""
    r = report
    tier_color = {'Low': '#27ae60', 'Moderate': '#f39c12', 'High': '#e74c3c'}
    color = tier_color.get(r['risk_summary']['tier'], 'gray')

    toxicity_rows = "".join(
        f"<tr><td>{'⚠️ ' if t['label']=='TOXIC' else ''}{name}</td>"
        f"<td>{t['probability']:.3f}</td>"
        f"<td style='color:{'red' if t['label']==\"TOXIC\" else \"green\"}'>{t['label']}</td></tr>"
        for name, t in r['toxicity'].items()
    )
    admet = r['admet']
    admet_rows = (
        f"<tr><td>Molecular Weight</td><td>{admet['molecular_weight']} Da</td></tr>"
        f"<tr><td>LogP</td><td>{admet['logp']}</td></tr>"
        f"<tr><td>H-Bond Donors</td><td>{admet['hbd']}</td></tr>"
        f"<tr><td>H-Bond Acceptors</td><td>{admet['hba']}</td></tr>"
        f"<tr><td>TPSA</td><td>{admet['tpsa']} Å²</td></tr>"
        f"<tr><td>Rotatable Bonds</td><td>{admet['rotatable_bonds']}</td></tr>"
        f"<tr><td>Lipinski Ro5</td><td>{'✓ Pass' if admet['lipinski_pass'] else '✗ Fail'}</td></tr>"
        f"<tr><td>Veber Rules</td><td>{'✓ Pass' if admet['veber_pass'] else '✗ Fail'}</td></tr>"
        f"<tr><td>PAINS Alerts</td><td>{', '.join(admet['pains_alerts']) or 'None'}</td></tr>"
    )
    html = f"""
    <div style="font-family:sans-serif;max-width:900px;border:1px solid #ddd;border-radius:8px;padding:16px">
      <h2>{r['compound_name'] or 'Compound'} — Safety Profile</h2>
      <p style="font-family:monospace;font-size:12px;color:#555">{r['canonical_smiles']}</p>
      <div style="display:flex;gap:24px;align-items:flex-start">
        <div>{r['structure_svg']}</div>
        <div>
          <div style="background:{color};color:white;padding:8px 16px;border-radius:6px;font-size:18px;font-weight:bold;margin-bottom:12px">
            Risk: {r['risk_summary']['tier']} ({r['risk_summary']['composite_score']:.3f})
          </div>
          <p>Toxic targets: {', '.join(r['risk_summary']['toxic_targets']) or 'None'}</p>
        </div>
      </div>
      <div style="display:flex;gap:24px;margin-top:16px">
        <div style="flex:1">
          <h3>Toxicity Predictions</h3>
          <table style="width:100%;border-collapse:collapse">
            <tr><th style="text-align:left">Target</th><th>Prob</th><th>Label</th></tr>
            {toxicity_rows}
          </table>
        </div>
        <div style="flex:1">
          <h3>ADMET Properties</h3>
          <table style="width:100%;border-collapse:collapse">
            {admet_rows}
          </table>
        </div>
      </div>
    </div>"""
    display(HTML(html))

In [ ]:
MOLECULES = {
    'Aspirin':      'CC(=O)Oc1ccccc1C(=O)O',
    'Tamoxifen':    'CCC(=C(c1ccccc1)c1ccc(OCCO)cc1)c1ccccc1',
    'Bisphenol A':  'CC(c1ccc(O)cc1)(c1ccc(O)cc1)',
    'Dioxin (TCDD)':'Clc1ccc2c(c1)Oc1cc(Cl)ccc1O2',
    'Caffeine':     'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
}

reports = {}
for name, smiles in MOLECULES.items():
    print(f"\n{'='*50}")
    print(f"Generating report: {name}")
    reports[name] = generate_report(smiles, name)
    render_report_html(reports[name])

In [ ]:
# Export Aspirin report as the API contract fixture
with open('sample_report.json', 'w') as f:
    json.dump(reports['Aspirin'], f, indent=2)
print("Saved sample_report.json — this is the API contract fixture for backend tests.")

# Verify it round-trips cleanly
with open('sample_report.json') as f:
    loaded = json.load(f)
assert loaded['smiles'] == reports['Aspirin']['smiles']
assert len(loaded['toxicity']) == 12
print("✓ sample_report.json round-trip verified")